# Retail Customer Behavior & Sales Intelligence
**Portfolio project — Python / Pandas / EDA**

This notebook independently implements the analysis workflow inspired by the referenced tutorial. It uses the supplied retail customer-shopping dataset and adds a clear business-focused analysis layer.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/customer_shopping_behavior_raw.csv')
df.head()

## 1. Data quality audit
Check shape, data types, missing values and duplicate records before analysis.

In [ ]:
print('Shape:', df.shape)
display(df.info())
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_values'))
print('Duplicate rows:', df.duplicated().sum())

## 2. Cleaning and feature engineering
The dataset contains 37 missing review ratings. We impute them with the overall median rating and create age/customer segments for business analysis.

In [ ]:
df.columns = [c.strip().lower().replace(' ','_').replace('(','').replace(')','').replace('-','_') for c in df.columns]
df = df.rename(columns={'purchase_amount_usd':'purchase_amount'})
df['review_rating'] = df['review_rating'].fillna(df['review_rating'].median())
df['age_group'] = pd.cut(
    df['age'], bins=[17,25,35,45,55,70],
    labels=['18-25','26-35','36-45','46-55','56-70'],
    include_lowest=True
)
df['customer_segment'] = np.select(
    [df['previous_purchases'].eq(1), df['previous_purchases'].between(2,10)],
    ['New','Returning'], default='Loyal'
)
df['discount_flag'] = np.where(df['discount_applied'].eq('Yes'),1,0)
df['promo_flag'] = np.where(df['promo_code_used'].eq('Yes'),1,0)
df.head()

## 3. KPI snapshot

In [ ]:
kpis = {
    'Orders': len(df),
    'Revenue (USD)': df['purchase_amount'].sum(),
    'Average Order Value (USD)': df['purchase_amount'].mean(),
    'Average Rating': df['review_rating'].mean(),
    'Subscriber Share': (df['subscription_status'].eq('Yes').mean()*100),
    'Discount Usage': (df['discount_flag'].mean()*100)
}
pd.Series(kpis).round(2)

## 4. Exploratory analysis

In [ ]:
category_summary = df.groupby('category').agg(
    revenue=('purchase_amount','sum'),
    orders=('customer_id','count'),
    avg_order=('purchase_amount','mean')
).sort_values('revenue', ascending=False)
display(category_summary.round(2))

age_summary = df.groupby('age_group', observed=True).agg(
    revenue=('purchase_amount','sum'),
    orders=('customer_id','count'),
    avg_order=('purchase_amount','mean')
)
display(age_summary.round(2))

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4))
category_summary['revenue'].plot(kind='bar', ax=axes[0], title='Revenue by Category')
age_summary['revenue'].plot(kind='bar', ax=axes[1], title='Revenue by Age Group')
plt.tight_layout()

## 5. Customer and product analysis

In [ ]:
subscription_summary = df.groupby('subscription_status').agg(
    customers=('customer_id','count'),
    revenue=('purchase_amount','sum'),
    avg_order=('purchase_amount','mean')
)
display(subscription_summary.round(2))

segment_summary = df.groupby('customer_segment').agg(
    customers=('customer_id','count'),
    revenue=('purchase_amount','sum'),
    avg_order=('purchase_amount','mean')
).sort_values('revenue', ascending=False)
display(segment_summary.round(2))

top_products = df.groupby('item_purchased').agg(
    orders=('customer_id','count'),
    revenue=('purchase_amount','sum'),
    avg_rating=('review_rating','mean')
).sort_values('revenue', ascending=False).head(10)
display(top_products.round(2))

## 6. Analyst takeaways
- Clothing is the largest revenue category in this dataset.
- The 56–70 age group contributes the largest revenue share.
- Most recorded customers are classified as Loyal using the project segmentation rule.
- Subscribers represent a minority of customers, so subscription conversion is a clear business metric to monitor.
- Discount usage is substantial; product-level discount rates should be monitored before broad promotions are expanded.
- The dataset has no transaction date, so genuine monthly/weekly revenue trends cannot be claimed from this source.